Convert embeddings ->

In [7]:
import pandas as pd
import numpy as np

In [8]:
df_videos=pd.read_csv("df_videos_text_and_sentiment.csv")
#df_comments=pd.read_parquet("merged_asof_comments_sentiment_relative_stock_diff.parquet")

In [10]:
#normlize white space in text_to_embed
import re

def clean_whitespace(text):
    # Replace all whitespace (spaces, tabs, newlines) with a single space
    text = re.sub(r'\s+', ' ', text)
    # Remove leading/trailing spaces
    text = text.strip()
    return text
df_videos['video_text'] = df_videos['video_text'].fillna("")
df_videos['video_text_normalized'] = df_videos['video_text'].apply(lambda x: clean_whitespace(x))
# df_comments['....']=df_comment['...'].apply(lambda x: clean_whitespace(x)

In [11]:
import torch
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [12]:
from tqdm import tqdm
import logging
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device=device
)

logging.basicConfig(filename="embeddings.log", filemode="a", format="%(asctime)s | %(levelname)s %(message)s")
logger = logging.getLogger()

def get_embeddings(texts):
    all_embeddings = []
    with torch.no_grad():
        embeddings = model.encode(texts,
        batch_size=5,          # keep minimal
        truncate_dim=256,
        convert_to_numpy=True,
        show_progress_bar=True
        )
    logger.info(f"Proceseed rows")

    return embeddings

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6319.49it/s]


In [13]:
#Embeddings for videos
df_videos['videos_text_embeddings']=df_videos['video_text_normalized'].apply(lambda x: np.array(get_embeddings(x), dtype=np.float32))
#Saving to Parquet
df_videos.to_parquet("df_videos_text_sentiment_embeddings.parquet")

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]
